Requirements

In [ ]:
%pip install -q kagglehub libreyolo
%pip install -q --upgrade jupyter ipywidgets
# !git clone https://github.com/LuisPeregrina/gdl-atsc-anti-spillback.git
# !mv gdl-atsc-anti-spillback/* .


Imports

In [ ]:
from pathlib import Path

import kagglehub
from libreyolo import LibreYOLO

from tools.mtid_split_yolo import split_dataset


Config

In [ ]:
MODEL_NAME = "LibreFOMOs-point"
DATASET_PATH = Path.cwd() / "dataset"
DATASET_NAME = "andreasmoegelmose/multiview-traffic-intersection-dataset"
EPOCHS = 2

# Configs per model
MODELS = {
    "LibreFOMOs-point": {
        "image_size": 96,
        "batch_size": -1,
    },
    "YOLOv9t": {
        "image_size": 640,
        "batch_size": -1,
    },
}
IMAGE_SIZE = MODELS[MODEL_NAME]["image_size"]
BATCH_SIZE = MODELS[MODEL_NAME]["batch_size"]


Dataset

In [ ]:

kagglehub.dataset_download(DATASET_NAME, output_dir=str(DATASET_PATH))
yaml_path = split_dataset(DATASET_PATH)
model = LibreYOLO(f"{MODEL_NAME}.pt")


Train

In [ ]:
model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
)
model.export(format="pt")

Validate

In [ ]:
metrics_val = model.val(data=yaml_path, split="val", batch=BATCH_SIZE)
print("Validation mAP50-95:", metrics_val["metrics/mAP50-95"])


Test

In [ ]:
# Test metrics (final evaluation)
metrics_test = model.val(data=yaml_path, split="test", batch=16)
print("Test mAP50-95:", metrics_test["metrics/mAP50-95"])

Results

In [ ]:
print("Validation metrics/mAP50-95:", metrics_val["metrics/mAP50-95"])
print("Validation speed/total_ms:", metrics_val["speed/total_ms"])
print("Test metrics/mAP50-95:", metrics_test["metrics/mAP50-95"])
print("Test speed/total_ms:", metrics_test["speed/total_ms"])

In [ ]:
import cv2

video_path = "samples/288312_tiny.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

while True:
    success, frame = cap.read()
    if not success:
        break

    results = model.predict(
        source=frame,
        imgsz=IMAGE_SIZE,
        conf=0.25,
        verbose=False,
    )

    result = results[0]
    boxes = getattr(result, "boxes", None)
    names = getattr(result, "names", getattr(model, "names", {}))

    if boxes is not None:
        coordinates = boxes.xyxy.tolist()
        confidences = boxes.conf.tolist()
        class_ids = boxes.cls.tolist()

        for box, confidence, class_id in zip(
            coordinates, confidences, class_ids
        ):
            class_id = int(class_id)
            class_name = names[class_id] if isinstance(names, (list, tuple)) else names.get(
                class_id, str(class_id)
            )

            if class_name.lower() not in {"car", "cars"}:
                continue

            x1, y1, x2, y2 = map(int, box)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"{class_name} {confidence:.2f}",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2,
            )

    cv2.imshow("Car Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()